In [ ]:
import wfdb
import numpy as np
import pandas as pd
import neurokit2 as nk
from scipy.signal import butter, filtfilt

In [ ]:
# Bandpass Filter - remove baseline wander and high-frequency noise

def bandpass_filter(signal, lowcut=0.5, highcut=40.0, fs=300, order=4):
    nyq = fs / 2
    low, high = lowcut / nyq, highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)

In [ ]:
# R-peak detection — using NeuroKit2

def detect_rpeaks(signal, fs=300):
    signals, info = nk.ecg_process(signal, sampling_rate=fs)
    return info['ECG_R_Peaks']

In [ ]:
# Convert R-peak sample indices to RR intervals in seconds

def compute_rr_intervals(rpeaks, fs=300):
    if len(rpeaks) < 2:
        return np.array([])
    return np.diff(rpeaks) / fs   # seconds between beats

In [ ]:
# Signal Segmentation — fixed-length windows of 30 seconds

def segment_signal(signal, fs=300, window_sec=30):
    window_len = fs * window_sec
    segments = []
    for start in range(0, len(signal) - window_len, window_len):
        segments.append(signal[start:start + window_len])
    return segments

In [ ]:
# Normalization — zero mean, unit variance per recording

def normalize_signal(signal):
    std = signal.std()
    if std == 0:
        return signal - signal.mean()
    return (signal - signal.mean()) / std

In [ ]:
labels_df = pd.read_csv('data/raw/cinc2017/REFERENCE-v3.csv',
                        header=None, names=['record', 'label'])

X_signal, X_rr, y = [], [], []

for _, row in labels_df.iterrows():
    record = wfdb.rdrecord(f"data/raw/cinc2017/{row['record']}")
    signal = record.p_signal[:, 0]

    # Step 1 — filter
    signal = bandpass_filter(signal)

    # Step 2 — R-peak detection → RR intervals
    try:
        rpeaks = detect_rpeaks(signal)
        rr = compute_rr_intervals(rpeaks)
    except Exception:
        rr = np.array([])

    # Step 3 — fixed-length segments
    segments = segment_signal(signal)

    # Step 4 — normalize each segment
    segments = [normalize_signal(s) for s in segments]

    for seg in segments:
        X_signal.append(seg)
        X_rr.append(rr)       # same RR sequence shared across segments of one recording
        y.append(row['label'])

X_signal = np.array(X_signal)
y = np.array(y)

# Step 5 — save
np.save('data/processed/X_signal.npy', X_signal)
np.save('data/processed/y.npy', y)
np.save('data/processed/X_rr.npy', np.array(X_rr, dtype=object))  # ragged array